# Audience Retention & Churn Predictor for OTT Platform
### End-to-End Data Science & Machine Learning Capstone Analysis
**Course:** Third-Year CS / Data Science | **Language:** Python 3.10+ | **Target:** Academic Presentation & Viva

## 1. Project Introduction & Objectives
Over-The-Top (OTT) media services face significant economic pressures from subscriber churn and episode drop-offs.
This notebook demonstrates a rigorous, reproducible workflow covering:
1. Data validation and data quality profiling
2. Exploratory Data Analysis (EDA) and inferential hypothesis testing via SciPy
3. Leakage-free preprocessing using Scikit-Learn Pipelines
4. Feature engineering for subscriber engagement and episodic friction
5. Supervised modeling & hyperparameter tuning for churn prediction
6. Episode drop-off modeling distinguishing between full post-completion and pre-completion Early-Warning signals
7. Unsupervised audience segmentation using K-Means and PCA
8. Global & local model explainability

In [ ]:
import os
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is on sys.path
sys.path.append(str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

from src.data_loader import load_churn_data, load_retention_data, summarize_data_quality
from src.feature_engineering import engineer_churn_features, engineer_retention_features
from src.metrics import perform_welch_ttest, perform_chisquare_test
from src.churn_model import prepare_churn_data, get_candidate_classifiers
from src.retention_model import prepare_retention_data
from src.segmentation import perform_segmentation
from src.explainability import compute_global_feature_importance

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 2. Dataset Ingestion & Schema Quality Assessment

In [ ]:
df_churn = load_churn_data()
df_ret = load_retention_data()

print(f"Churn Dataset: {df_churn.shape[0]:,} rows, {df_churn.shape[1]} columns")
print(f"Retention Dataset: {df_ret.shape[0]:,} rows, {df_ret.shape[1]} columns")

print("\n--- Data Quality: Churn Dataset ---")
print(summarize_data_quality(df_churn, 'netflix_churn'))

print("\n--- Target Distribution (Churned) ---")
display(df_churn['churned'].value_counts(normalize=True).rename({0: 'Retained', 1: 'Churned'}))

print("\n--- Target Distribution (Drop-off) ---")
display(df_ret['drop_off'].value_counts(normalize=True).rename({0: 'Completed', 1: 'Drop-off'}))

## 3. Exploratory Data Analysis & Statistical Hypothesis Testing
We investigate whether watch volume and inactivity exhibit statistically significant differences across churn cohorts via Welch's t-test, and test categorical dependency via Pearson's Chi-Square test.

In [ ]:
retained_watch = df_churn[df_churn['churned'] == 0]['watch_hours']
churned_watch = df_churn[df_churn['churned'] == 1]['watch_hours']

t_res_watch = perform_welch_ttest(retained_watch, churned_watch, 'Watch Hours')
print("Welch's t-test for Watch Hours:")
print(json.dumps(t_res_watch, indent=2))

chi2_sub = perform_chisquare_test(df_churn, 'subscription_type', 'churned')
print("\nChi-Square test for Subscription Type:")
print(json.dumps(chi2_sub, indent=2))

## 4. Feature Engineering & Normalization
We compute the composite **Subscriber Engagement Score** and episode **Friction Index**.

In [ ]:
df_churn_feat = engineer_churn_features(df_churn)
df_ret_feat = engineer_retention_features(df_ret)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df_churn_feat, x='engagement_score', hue='churned', kde=True, ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Engineered Engagement Score Distribution')

sns.boxplot(data=df_ret_feat, x='drop_off', y='friction_index', ax=axes[1], palette=['#2ecc71', '#e74c3c'])
axes[1].set_title('Episode Friction Index vs Drop-Off')
plt.tight_layout()
plt.show()

## 5. Subscriber Churn Model Benchmarking
Comparing Dummy Baseline, Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting with Stratified 5-Fold CV.

In [ ]:
with open('models/churn_metrics.json') as f:
    churn_benchmarks = json.load(f)

print(f"Champion Model: {churn_benchmarks['best_model_name']}")
results_table = []
for name, m in churn_benchmarks['all_model_benchmarks'].items():
    results_table.append({
        'Model': name,
        'Accuracy': m['accuracy'],
        'Precision': m['precision'],
        'Recall': m['recall'],
        'F1-Score': m['f1_score'],
        'ROC-AUC': m.get('roc_auc')
    })
display(pd.DataFrame(results_table))

## 6. Episode Retention: Full vs Early-Warning Model
We investigate whether watch completion percentage induces leakage, evaluating a separate Early-Warning model with only pre-completion content characteristics.

In [ ]:
with open('models/retention_metrics.json') as f:
    ret_benchmarks = json.load(f)

for mode in ['full', 'early']:
    print(f"=== Retention Framework: {mode.upper()} ===")
    champ = ret_benchmarks[mode]['champion_name']
    m = ret_benchmarks[mode]['metrics'][champ]
    print(f"Champion: {champ} | Accuracy: {m['accuracy']} | F1: {m['f1_score']} | ROC-AUC: {m['roc_auc']} | Recall: {m['recall']}")

## 7. Audience Segmentation (K-Means & PCA)

In [ ]:
with open('models/segmentation_metrics.json') as f:
    seg_data = json.load(f)

print(f"Selected k: {seg_data['selected_k']}")
print(f"PCA Explained Variance: {seg_data['pca_explained_variance_ratio']}")

prof_df = pd.DataFrame(seg_data['profiles']).T
display(prof_df[['cluster_name', 'percentage', 'churn_rate', 'watch_hours_mean', 'last_login_days_mean', 'monthly_fee_mean']])

## 8. Model Explainability & Recommendations

In [ ]:
import joblib
from src.explainability import compute_global_feature_importance
from src.recommendations import generate_subscriber_recommendations

churn_pipe = joblib.load('models/churn_pipeline.joblib')
X_train, X_test, y_train, y_test, feat_cols = prepare_churn_data(df_churn)
imp_results = compute_global_feature_importance(churn_pipe, X_test, y_test, top_n=10)

print("Global Feature Importances:")
for feat, val in imp_results['native_importances'].items():
    print(f" - {feat:<25}: {val:.4f}")

# Sample recommendation for an at-risk subscriber
sample_user = {
    'monthly_fee': 13.99,
    'last_login_days': 26,
    'watch_hours': 4.5,
    'payment_method': 'Gift Card',
    'number_of_profiles': 1
}
rec = generate_subscriber_recommendations(0.88, sample_user)
print("\n--- Sample Retention Recommendation ---")
print(f"Risk Tier: {rec['risk_tier']} | Monthly Revenue at Risk: ${rec['monthly_revenue_at_risk']}")
for a in rec['recommended_interventions']:
    print(f"Action: {a['action']} -> {a['rationale']}")

## 9. Conclusion & Limitations
- **Synthetic Data Disclaimer:** The datasets contain synthetic behavioral attributes; models predict synthetic patterns for academic demonstration.
- **Predictive Champions:** Gradient Boosting provides exceptional discriminative power on both subscriber churn and episode drop-off.
- **Actionability:** Early-warning episode models allow creators to forecast retention risk before viewership numbers finalize.